# 🔤 OCR Worker — AIC 2026
**Chạy trên Colab GPU T4 (miễn phí)**

## Cách dùng:
1. Đặt `PART` và `TOTAL` theo phân công của nhóm
2. Upload data BTC lên Google Drive
3. Chạy từng cell từ trên xuống
4. Download file `ocr_part_X_of_Y.json` về gửi cho người merge

In [ ]:
# ═══════════════════════════════════════════
# ⚙️  CẤU HÌNH — Chỉnh tại đây trước khi chạy
# ═══════════════════════════════════════════

PART  = 1   # Phần của bạn (1, 2, hoặc 3)
TOTAL = 3   # Tổng số người

# Nhánh Git của bạn
GIT_REPO  = 'https://github.com/YOUR_USERNAME/AI_Challenge_2026.git'
GIT_BRANCH = 'ten-nhanh-ca-nhan-cua-ban'   # vd: 'feature/ocr-worker'

# Thư mục data BTC trên Google Drive
# Cấu trúc Drive cần có: MyDrive/AIC2026_data/btc/Keyframes/<video_id>/*.jpg
DRIVE_DATA_PATH = '/content/drive/MyDrive/AIC2026_data'

## Bước 1 — Mount Google Drive & Clone code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
import os

# Clone nhánh cá nhân
if not os.path.exists('/content/AIC_2026'):
    !git clone -b {GIT_BRANCH} {GIT_REPO} /content/AIC_2026
else:
    !cd /content/AIC_2026 && git pull origin {GIT_BRANCH}

os.chdir('/content/AIC_2026')
print('✅ Code ready at /content/AIC_2026')

## Bước 2 — Trỏ data từ Drive vào project (symlink, không copy)
> **Lưu ý về cấu trúc Drive:**  
> `MyDrive/AIC2026_data/btc/Keyframes/L21_V001/*.jpg`  
> `MyDrive/AIC2026_data/btc/Keyframes/L21_V002/*.jpg`  
> ...v.v.

In [ ]:
import os
from pathlib import Path

drive_btc   = Path(DRIVE_DATA_PATH) / 'btc'
project_btc = Path('/content/AIC_2026/data/btc')

# Xoá symlink cũ nếu có
if project_btc.is_symlink() or project_btc.exists():
    project_btc.unlink() if project_btc.is_symlink() else None

# Tạo symlink Drive → project (không tốn dung lượng Colab)
project_btc.parent.mkdir(parents=True, exist_ok=True)
if not project_btc.exists():
    os.symlink(str(drive_btc), str(project_btc))

# Kiểm tra
kf_dir = project_btc / 'Keyframes'
if kf_dir.exists():
    video_dirs = list(kf_dir.iterdir())
    print(f'✅ Tìm thấy {len(video_dirs)} thư mục video trong {kf_dir}')
else:
    print(f'❌ Không tìm thấy: {kf_dir}')
    print(f'   Kiểm tra lại DRIVE_DATA_PATH = "{DRIVE_DATA_PATH}"')
    print(f'   Cấu trúc Drive cần: {DRIVE_DATA_PATH}/btc/Keyframes/<video_id>/*.jpg')

## Bước 3 — Cài EasyOCR

In [ ]:
!pip install easyocr -q
print('✅ EasyOCR installed')

## Bước 4 — Xem kế hoạch chia (tuỳ chọn)

In [ ]:
!python -m src.indexing.run_ocr_worker --plan --total {TOTAL}

## Bước 5 — Chạy OCR
> ⏱️ Ước tính thời gian với **Colab GPU T4**: ~5-10h tùy số video
> 
> Script **tự resume** nếu session bị ngắt — chạy lại cell này là tiếp tục.

In [ ]:
!python -m src.indexing.run_ocr_worker --part {PART} --total {TOTAL}

## Bước 6 — Copy output về Drive để không mất khi session hết

In [ ]:
import shutil
from pathlib import Path

src_file = Path(f'/content/AIC_2026/data/processed/ocr_part_{PART}_of_{TOTAL}.json')
dst_dir  = Path(DRIVE_DATA_PATH) / 'ocr_results'
dst_dir.mkdir(parents=True, exist_ok=True)

if src_file.exists():
    dst_file = dst_dir / src_file.name
    shutil.copy2(src_file, dst_file)
    size_mb = dst_file.stat().st_size / 1024 / 1024
    print(f'✅ Đã copy: {dst_file} ({size_mb:.1f} MB)')
    print(f'\n📥 Download file về máy từ Google Drive:')
    print(f'   Drive path: MyDrive/AIC2026_data/ocr_results/{src_file.name}')
else:
    print(f'❌ Chưa có file: {src_file}')

## Bước 7 — Merge (chỉ người được phân công merge chạy cell này)
> Copy tất cả file `ocr_part_*.json` vào `data/processed/` trước khi merge

In [ ]:
# Copy tất cả file partial từ Drive về
import shutil
from pathlib import Path

src_dir = Path(DRIVE_DATA_PATH) / 'ocr_results'
dst_dir = Path('/content/AIC_2026/data/processed')

for f in src_dir.glob('ocr_part_*.json'):
    shutil.copy2(f, dst_dir / f.name)
    print(f'  Copied: {f.name}')

# Merge
!python -m src.indexing.run_ocr_worker --merge

# Copy kết quả merge về Drive
merged = Path('/content/AIC_2026/data/processed/ocr_index.json')
if merged.exists():
    shutil.copy2(merged, Path(DRIVE_DATA_PATH) / 'ocr_index.json')
    print(f'\n✅ ocr_index.json đã được lưu vào Drive!')